In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

from nba_crunch.data_processing import enrich_free_throws, season_from_game_id

pd.set_option('display.max_columns', None)

sample = pd.read_csv('../data/raw/pbp/0022300001.csv', dtype={'gameId': str})
print(sample.columns.tolist())
sample.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022300001,2,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.0,0.0,0,NaN,Start of 1st Period (7:11 PM EST),period,start,0,0,1
1,0022300001,4,PT12M00.00S,1,1610612754,IND,1626167,Turner,M. Turner,0,0,0,NaN,0,NaN,NaN,0,h,Jump Ball Turner vs. Allen: Tip to Toppin,Jump Ball,NaN,1,0,2
2,0022300001,7,PT11M41.00S,1,1610612754,IND,1626167,Turner,M. Turner,2,21,2,Made,1,2.0,0.0,2,h,Turner 2' Cutting Dunk Shot (2 PTS) (Haliburto...,Made Shot,Cutting Dunk Shot,1,2,3
3,0022300001,9,PT11M23.00S,1,1610612739,CLE,1630596,Mobley,E. Mobley,59,53,8,Missed,1,NaN,NaN,0,v,MISS Mobley 8' Turnaround Jump Shot,Missed Shot,Turnaround Jump Shot,1,2,4
4,0022300001,10,PT11M20.00S,1,1610612754,IND,1626167,Turner,M. Turner,0,0,0,NaN,0,NaN,NaN,0,h,Turner REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5


In [4]:
print(season_from_game_id('0022300001'))  # expect '2023-24'
print(season_from_game_id('0022400001'))  # expect '2024-25'
print(season_from_game_id('0022500001'))  # expect '2025-26'

2023-24
2024-25
2025-26


In [5]:
pbp_dir = Path('../data/raw/pbp')
all_files = list(pbp_dir.glob('*.csv'))
print(f"Found {len(all_files)} files")

dfs = []
errors = []

for filepath in tqdm(all_files):
    try:
        df = pd.read_csv(filepath, dtype={'gameId': str, 'scoreHome': str, 'scoreAway': str})
        df_enriched = enrich_free_throws(df)
        df_enriched['season'] = season_from_game_id(df_enriched['gameId'].iloc[0])
        dfs.append(df_enriched)
    except Exception as e:
        errors.append((filepath.name, str(e)))

all_fts = pd.concat(dfs, ignore_index=True)
print(f"Total FTs: {len(all_fts)}")
print(f"Errors: {len(errors)}")

Found 3690 files


100%|██████████| 3690/3690 [00:10<00:00, 366.06it/s]


Total FTs: 164510
Errors: 0


In [7]:
print(all_fts.columns.tolist())
print(all_fts.shape)
all_fts.head()

['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId', 'clock_seconds', 'made', 'margin', 'is_crunch', 'season']
(164510, 29)


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,clock_seconds,made,margin,is_crunch,season
0,0022300688,14,PT10M29.00S,1,1610612746,LAC,202695,Leonard,K. Leonard,0,0,0,NaN,0,4,3,7,v,Leonard Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,0,9,629.0,True,1,False,2023-24
1,0022300688,15,PT10M29.00S,1,1610612746,LAC,202695,Leonard,K. Leonard,0,0,0,NaN,0,4,4,8,v,Leonard Free Throw 2 of 2 (2 PTS),Free Throw,Free Throw 2 of 2,1,0,10,629.0,True,0,False,2023-24
2,0022300688,21,PT10M14.00S,1,1610612765,DET,1631105,Duren,J. Duren,0,0,0,NaN,0,7,4,11,h,Duren Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,1,0,15,614.0,True,3,False,2023-24
3,0022300688,88,PT04M44.00S,1,1610612746,LAC,202695,Leonard,K. Leonard,0,0,0,NaN,0,22,11,33,v,Leonard Free Throw 1 of 1 (5 PTS),Free Throw,Free Throw 1 of 1,1,0,65,284.0,True,11,False,2023-24
4,0022300688,113,PT02M39.00S,1,1610612765,DET,201568,Gallinari,D. Gallinari,0,0,0,NaN,0,27,19,46,h,Gallinari Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,0,82,159.0,True,8,False,2023-24


In [8]:
print(f"Total FTs: {len(all_fts)}")
print(f"Crunch FTs: {all_fts['is_crunch'].sum()}")
print(f"Crunch share: {all_fts['is_crunch'].mean():.2%}")

Total FTs: 164510
Crunch FTs: 9402
Crunch share: 5.72%


In [9]:
print(all_fts['subType'].unique())

<StringArray>
[           'Free Throw 1 of 2',            'Free Throw 2 of 2',
            'Free Throw 1 of 1',         'Free Throw Technical',
            'Free Throw 1 of 3',            'Free Throw 2 of 3',
            'Free Throw 3 of 3',   'Free Throw Flagrant 1 of 2',
   'Free Throw Flagrant 2 of 2',   'Free Throw Flagrant 1 of 1',
   'Free Throw Flagrant 1 of 3',   'Free Throw Flagrant 2 of 3',
   'Free Throw Flagrant 3 of 3', 'Free Throw Clear Path 1 of 2',
 'Free Throw Clear Path 2 of 2',  'Free Throw Technical 1 of 2',
  'Free Throw Technical 2 of 2']
Length: 17, dtype: str


In [10]:
all_fts.to_csv('../data/processed/all_fts.csv', index=False)

In [11]:
crunch = all_fts[all_fts['is_crunch']]
non_crunch = all_fts[~all_fts['is_crunch']]

print(f"Non-crunch FT%: {non_crunch['made'].mean():.4f}  (n = {len(non_crunch):,})")
print(f"Crunch FT%:     {crunch['made'].mean():.4f}  (n = {len(crunch):,})")
print(f"Difference:     {(crunch['made'].mean() - non_crunch['made'].mean()):.4f}")

Non-crunch FT%: 0.7828  (n = 155,108)
Crunch FT%:     0.7746  (n = 9,402)
Difference:     -0.0082


In [14]:
len(crunch)

9402

In [20]:
from statsmodels.stats.proportion import proportions_ztest

# Compute the four numbers you need
crunch_makes = crunch['made'].sum()
crunch_n = len(crunch)
non_crunch_makes = non_crunch['made'].sum()
non_crunch_n = len(non_crunch)

# Package into the format the function expects (two arrays: counts and totals)
counts = np.array([crunch_makes, non_crunch_makes])
totals = np.array([crunch_n, non_crunch_n])

# Call the function — returns (z_statistic, p_value)
z_stat, p_value = proportions_ztest(counts, totals)

# Print results
print(f"Crunch FT%:     {crunch['made'].mean():.4f}  ({crunch_makes} / {crunch_n})")
print(f"Non-crunch FT%: {non_crunch['made'].mean():.4f}  ({non_crunch_makes} / {non_crunch_n})")
print(f"z-statistic: {z_stat:.4f}")
print(f"p-value:     {p_value:.4f}")

Crunch FT%:     0.7746  (7283 / 9402)
Non-crunch FT%: 0.7828  (121418 / 155108)
z-statistic: -1.8650
p-value:     0.0622


Pooled across 164,510 free throws over three seasons, crunch-time FT% (77.46%, n=9,402) was 0.82 percentage points lower than non-crunch FT% (78.28%, n=155,108). A two-proportion z-test yielded p = 0.062, providing weak-to-moderate evidence against the null hypothesis of no difference. The magnitude is small enough that it would be unlikely to inform coaching decisions, even if real.

In [22]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=crunch_makes,
    nobs1=crunch_n,
    count2=non_crunch_makes,
    nobs2=non_crunch_n,
    method='wald',
    alpha=0.05  # 1 - confidence_level. 0.05 → 95% CI
)

print(f"Difference (crunch - non-crunch): {(crunch['made'].mean() - non_crunch['made'].mean()):.4f}")
print(f"95% CI: [{ci_low:.4f}, {ci_high:.4f}]")


Difference (crunch - non-crunch): -0.0082
95% CI: [-0.0169, 0.0005]
